In [ ]:
#Coefficient for LHQ (cluster 1) from ASICS
A = [-1.730215,-2.744504,-3.396561]
B = [1.251296,2.561384,-1.133846]
C = [3.573296,0.986618,-0.473004]
D = [-0.060956,-0.376258,-0.980912]
E = [-0.037878,-0.579924,-1.166911]

q = (3-4*Ispin*(Ispin + 1))/(16*wkhz)

# Function to solve the system of equations
# Function to solve the system of equations
def equations(vars, D, E, q):
    Axx, Ayy, Axy, Axz, Ayz = vars
    Azz = -(Axx + Ayy)  # Enforce the constraint directly

    # Equation system for Dx, Dy, Dz, Ex, Ey, Ez
    Dx = -((Ayy - Azz)**2 - 4*(Ayz)**2)*(9*q/8)
    Ex = Ayz*(Azz - Ayy)*(9*q/2)

    Dy = -((Axx - Azz)**2 - 4*(Axz)**2)*(9*q/8)
    Ey = Axz*(Azz - Axx)*(9*q/2)

    Dz = -((Axx - Ayy)**2 - 4*(Axy)**2)*(9*q/8)
    Ez = Axy*(Ayy - Axx)*(9*q/2)

    # Debugging prints
    # print(f"Axx: {Axx}, Ayy: {Ayy}, Azz: {Azz}, Axy: {Axy}, Axz: {Axz}, Ayz: {Ayz}")
    # print(f"Calculated Dx: {Dx}, Dy: {Dy}, Dz: {Dz}, Ex: {Ex}, Ey: {Ey}, Ez: {Ez}")


    # Equations to solve
    eq1 = Dx - D[0]
    eq2 = Ex - E[0]

    eq3 = Dy - D[1]
    eq4 = Ey - E[1]

    eq5 = Dz - D[2]
    eq6 = Ez - E[2]

    # Return only the first 5 equations as fsolve expects
    return [eq1, eq2, eq3, eq4, eq5]

#initial guess A_xx, A_yy, A_xy, A_xz, A_yz
initial_guess = [1, 1, 2, 2, 1]
# Solve the system
solution = fsolve(equations, initial_guess, args=(D, E, q))

# Extract the solutions
Axx, Ayy, Axy, Axz, Ayz = solution
Azz = -(Axx + Ayy)

# Print the results
print(f"Axx: {Axx}")
print(f"Ayy: {Ayy}")
print(f"Azz: {Azz}")
print(f"Axy: {Axy}")
print(f"Axz: {Axz}")
print(f"Ayz: {Ayz}")

# Ensure the calculated Ez matches the given E[2]
Ez = Axy * (Ayy - Axx) * (9 * q / 2)
print(f"Calculated Ez: {Ez}, Expected Ez: {E[2]}")


In [ ]:
Dx = -((Ayy - Azz)**2 - 4*(Ayz)**2)*(9*q/8)
Ex = Ayz*(Azz - Ayy)*(9*q/2)

Dy = -((Axx - Azz)**2 - 4*(Axz)**2)*(9*q/8)
Ey = Axz*(Azz - Axx)*(9*q/2)

Dz = -((Axx - Ayy)**2 - 4*(Axy)**2)*(9*q/8)
Ez = Axy*(Ayy - Axx)*(9*q/2)

print(Dx, Dy, Dz, Ex, Ey, Ez)

In [ ]:
Q_T = np.zeros((3,3))
Q_T[0,0] = Axx; Q_T[0,1] = Axy; Q_T[0,2] = Axz
Q_T[1,0] = Axy; Q_T[1,1] = Ayy; Q_T[1,2] = Ayz;
Q_T[2,0] = Axz; Q_T[2,1] = Ayz; Q_T[2,2] = Azz;
print('Quadrupolar Tensor in Tenon frame: \n', Q_T)

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(Q_T)
D_quad = np.diag(eigenvalues)
print('Diagonalized Quadrupolar Tensor:\n', D_quad)
quad_avg = np.mean(eigenvalues)
sorted_eigenvalues = sorted((eigenvalues - quad_avg), key=abs)

Gzz_q = (sorted_eigenvalues[2] + quad_avg)*(2*Ispin*(2*Ispin - 1)) #following the Voseggard paper for principal frame parameters
Gxx_q = (sorted_eigenvalues[1] + quad_avg)*(2*Ispin*(2*Ispin - 1))
Gyy_q = (sorted_eigenvalues[0]+ quad_avg)*(2*Ispin*(2*Ispin - 1))

CQ_fit = Gzz_q/10**3
Qeta_fit = (Gyy_q - Gxx_q)/Gzz_q

print(CQ_fit, Qeta_fit)

In [ ]:
#Define symbol and force them to be real
AzzmAyy,AzzmAxx,AyymAxx,Ayz,Axz,Axy = sym.symbols('AzzmAyy,AzzmAxx,AyymAxx,Ayz,Axz,Axy', real=True)

#define set of equations
eq1 = sym.Eq(-((AzzmAyy)**2 - 4*(Ayz)**2)*((9*q/8)), D[0])
eq2 = sym.Eq(Ayz*(AzzmAyy)*(9*q/2), E[0])

eq3 = sym.Eq(-((AzzmAxx)**2 - 4*(Axz)**2)*(9*q/8), D[1])
eq4 = sym.Eq(Axz*(AzzmAxx)*(9*q/2), E[1])

eq5 = sym.Eq(-((AyymAxx)**2 - 4*(Axy)**2)*(9*q/8), D[2])
eq6 = sym.Eq(Axy*(AyymAxx)*(9*q/2), E[2])


# Solve the system
solution1 = sym.solve([eq1,eq2], (AzzmAyy,Ayz))

# Print the solutions
print(f"(Azz - Ayy) & Ayz: {solution1}")

solution2 = sym.solve([eq3,eq4], (AzzmAxx,Axz))

# Print the solutions
print(f"(Azz - Axx) & Axz: {solution2}")

solution3 = sym.solve([eq5,eq6], (AyymAxx,Axy))

# Print the solutions
print(f"(Ayy - Axx) & Axy: {solution3}")

AzzmAyy = [solution1[0][0], solution1[1][0]]
Ayz = [solution1[0][1], solution1[1][1]]
AzzmAxx = [solution2[0][0], solution2[1][0]]
Axz = [solution2[0][1], solution2[1][1]]
AyymAxx = [solution3[0][0], solution3[1][0]]
Axy = [solution3[0][1], solution3[1][1]]


# print(AzzmAyy, Ayz, AyymAxx)




In [ ]:
#Coefficient for LHQ (cluster 1) from ASICS
A = [-1.730215,-2.744504,-3.396561]
B = [1.251296,2.561384,-1.133846]
C = [3.573296,0.986618,-0.473004]
D = [-0.060956,-0.376258,-0.980912]
E = [-0.037878,-0.579924,-1.166911]

#factor q given in article
q = (3-4*Ispin*(Ispin + 1))/(16*wkhz)

#Function using fit parameters and q
def get_quad_tensor(fit_D,fit_E,q_value):

    #Define symbol and force them to be real
    AzzmAyy,AzzmAxx,AyymAxx,Ayz,Axz,Axy = sym.symbols('AzzmAyy,AzzmAxx,AyymAxx,Ayz,Axz,Axy', real=True)

    #Variable for each equation set
    quad_tensor = [(AzzmAyy, Ayz), (AzzmAxx, Axz), (AyymAxx, Axy)]

    # List to hold solutions
    solutions = []

    for i, (diagonal_diff, off_diagonal) in enumerate(quad_tensor):
        eq1 = sym.Eq(-((diagonal_diff)**2 - 4*(off_diagonal)**2)*((9*q_value/8)), fit_D[i])
        eq2 = sym.Eq(off_diagonal*(diagonal_diff)*(9*q_value/2), fit_E[i])

        # Solve the system
        solution = sym.solve([eq1, eq2], (diagonal_diff, off_diagonal))
        solutions.append(solution)

        #    # Print the solutions
        # print(f"Set {i+1}: (Difference Variable & Off-diagonal): {solution}")

    # Assign the solutions to the respective variables
    AzzmAyy = [solutions[0][0][0], solutions[0][1][0]]
    Ayz = [solutions[0][0][1], solutions[0][1][1]]
    AzzmAxx = [solutions[1][0][0], solutions[1][1][0]]
    Axz = [solutions[1][0][1], solutions[1][1][1]]
    AyymAxx = [solutions[2][0][0], solutions[2][1][0]]
    Axy = [solutions[2][0][1], solutions[2][1][1]]

    return (AzzmAyy, Ayz, AzzmAxx, Axz, AyymAxx, Axy)

def get_best_combination(AzzmAxx, AyymAxx, AzzmAyy):
    Axx1 = []; Axx2 = []; Axx3 = []
    Ayy1 = []; Ayy2 = []; Ayy3 = []
    Azz1 = []; Azz2 = []; Azz3 = []
    
    combinations = list(itertools.product(AzzmAxx, AyymAxx, AzzmAyy))
    # Initialize variables to get the best combination and minimum variation
    best_combination = None
    min_variation_Axx = float('inf')
    min_variation_Ayy = float('inf')
    min_variation_Azz = float('inf')
    
    for (AzzmAxx_value, AyymAxx_value, AzzmAyy_value) in combinations:
        # Solution 1
        Axx1_value = -(AzzmAxx_value + AyymAxx_value)/3
        Ayy1_value = Axx1_value + AyymAxx_value
        Azz1_value = Axx1_value + AzzmAxx_value

        #Save combinations
        Axx1.append(Axx1_value)
        Ayy1.append(Ayy1_value)
        Azz1.append(Azz1_value)

        # Solution 2
        Ayy2_value = -(AzzmAyy_value - AyymAxx_value)/3
        Axx2_value = Ayy2_value - AyymAxx_value
        Azz2_value = Ayy2_value + AzzmAyy_value

        #Save combinations
        Axx2.append(Axx2_value)
        Ayy2.append(Ayy2_value)
        Azz2.append(Azz2_value)

        # Solution 3
        Azz3_value = (AzzmAxx_value + AzzmAyy_value)/3
        Axx3_value = Azz3_value - AzzmAxx_value
        Ayy3_value = Azz3_value - AzzmAyy_value

        #Save combinations
        Axx3.append(Axx3_value)
        Ayy3.append(Ayy3_value)
        Azz3.append(Azz3_value)


    Axx = Axx1 + Axx2 + Axx3
    Ayy = Ayy1 + Ayy2 + Ayy3
    Azz = Azz1 + Azz2 + Azz3
    

    # Calculate variation (standard deviation) for Axx, Ayy, Azz
    variation_Axx = np.std(Axx)
    variation_Ayy = np.std(Ayy)
    variation_Azz = np.std(Azz)

    # Update the best combination if the current one has less variation
    if variation_Axx < min_variation_Axx:
        min_variation_Axx = variation_Axx
        # best_combination = (AzzmAxx_value,AyymAxx_value,AzzmAyy_value)
        best_avg_Axx = np.mean(Axx)

    if variation_Ayy < min_variation_Ayy:
        min_variation_Ayy = variation_Ayy
        # best_combination = (AzzmAxx_value,AyymAxx_value,AzzmAyy_value)
        best_avg_Ayy = np.mean(Ayy)
    
    if variation_Azz < min_variation_Azz:
        min_variation_Azz = variation_Azz
        # best_combination = (AzzmAxx_value,AyymAxx_value,AzzmAyy_value)
        best_avg_Azz = np.mean(Azz)
        
    return best_combination, best_avg_Axx, best_avg_Ayy, best_avg_Azz
   

    
        


AzzmAyy, Ayz, AzzmAxx, Axz, AyymAxx, Axy = get_quad_tensor(D, E, q)


best_combination, best_avg_Axx, best_avg_Ayy, best_avg_Azz = get_best_combination(get_quad_tensor(D, E, q))


# print("Best combination with minimum standard deviation:")
# print("AzzmAxx:", best_combination[0])
# print("AyymAxx:", best_combination[1])
# print("AzzmAyy:", best_combination[2])
# print("Average of Axx1, Axx2, Axx3 with minimum standard deviation:", best_avg_Axx)
# print("Average of Ayy1, Ayy2, Ayy3 with minimum standard deviation:", best_avg_Ayy)
# print("Average of Azz1, Azz2, Azz3 with minimum standard deviation:", best_avg_Azz)